## **Inference Notebook for Clustering and Classification using Credit Card Spending Data**

#### By: John Adrian T. Ada

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import math

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, precision_recall_curve, log_loss, roc_auc_score)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline

import mlflow
import mlflow.sklearn
mlflow.sklearn.autolog()
#mlflow server --port 8080

import sys
from pathlib import Path
import os

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

C:\Users\adaad\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from scripts.label_data import umap_reduce, dim_reduc_umap, kmeans_label
from scripts.load_preprocess_data import load_data, validate_and_clean_data, preprocess_data 

---
### **4. Multiple Model Training for Classification**


In [3]:
#Set aside a final validation set
main_df = pd.read_csv(r'../cc-dataset/CC GENERAL.csv')

train_df, validation_df = train_test_split(main_df, test_size=0.2, random_state=42, shuffle=True)
print(train_df.shape)
print(validation_df.shape)

(7160, 18)
(1790, 18)


In [4]:
#label the training data
train_df = preprocess_data(train_df)
reduced_df = dim_reduc_umap(train_df)
labeled_train_df = kmeans_label(reduced_df, train_df)

labeled_train_df

Dropped 1858 rows where MINIMUM_PAYMENTS >= PAYMENTS
Dropped 258 rows with empty fields


2026/09/23 12:52:35 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6cdd2a5e2ad84066894d6af73d566703', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 12:52:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/23 12:52:44 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.


,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE,kmeans_labels
7055,166.514439,0.777778,503.92,370.30,133.62,0.000000,0.777778,0.444444,0.666667,0.000000,0,14,2500.0,142.308045,130.127411,0.000000,9,5
8545,528.463885,1.000000,0.00,0.00,0.00,5191.738847,0.000000,0.000000,0.000000,0.833333,11,0,1000.0,4736.808799,309.547058,0.833333,12,2
4463,24.831137,0.636364,1909.13,30.00,1879.13,0.000000,0.916667,0.083333,0.833333,0.000000,0,12,2000.0,1831.935279,21.095616,0.000000,12,1
4011,137.864267,0.272727,237.24,0.00,237.24,1113.534935,1.000000,0.000000,1.000000,0.083333,2,12,1200.0,1232.901419,109.853069,0.000000,12,1
2906,34.869037,1.000000,356.32,0.00,356.32,0.000000,1.000000,0.000000,0.875000,0.000000,0,8,1000.0,286.841222,129.155038,0.571429,8,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,9699.252383,1.000000,0.00,0.00,0.00,10630.321020,0.000000,0.000000,0.000000,0.833333,27,0,10000.0,13001.037150,3897.073997,0.000000,12,2
466,2070.069187,1.000000,1382.57,1077.75,304.82,0.000000,1.000000,0.666667,0.833333,0.000000,0,33,2400.0,1580.081500,466.772558,0.000000,12,0
5734,1079.097023,1.000000,247.44,0.00,247.44,0.000000,1.000000,0.000000,1.000000,0.000000,0,12,1200.0,503.445403,302.481716,0.000000,12,0
5390,40.247238,0.181818,0.00,0.00,0.00,909.480894,0.000000,0.000000,0.000000,0.083333,2,0,1500.0,3578.648701,69.271137,1.000000,12,1


### **Class Imbalance Assessment**
---

In [17]:
#check the proportions of the classes
print(labeled_train_df['kmeans_labels'].value_counts())
print(len(labeled_train_df))


kmeans_labels
0    1726
3     881
1     825
4     655
2     612
5     345
Name: count, dtype: int64
5044


### **Model Training (baseline, no resampling methods)**
---

In [6]:
X = labeled_train_df.drop('kmeans_labels', axis=1)
y = labeled_train_df['kmeans_labels']

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42, 
                                                    shuffle=True, 
                                                    stratify=y)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (4035, 17), y_train shape: (4035,)
X_test shape: (1009, 17), y_test shape: (1009,)


In [7]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4035 entries, 968 to 4804
Data columns (total 17 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   BALANCE                           4035 non-null   float64
 1   BALANCE_FREQUENCY                 4035 non-null   float64
 2   PURCHASES                         4035 non-null   float64
 3   ONEOFF_PURCHASES                  4035 non-null   float64
 4   INSTALLMENTS_PURCHASES            4035 non-null   float64
 5   CASH_ADVANCE                      4035 non-null   float64
 6   PURCHASES_FREQUENCY               4035 non-null   float64
 7   ONEOFF_PURCHASES_FREQUENCY        4035 non-null   float64
 8   PURCHASES_INSTALLMENTS_FREQUENCY  4035 non-null   float64
 9   CASH_ADVANCE_FREQUENCY            4035 non-null   float64
 10  CASH_ADVANCE_TRX                  4035 non-null   int64  
 11  PURCHASES_TRX                     4035 non-null   int64  
 12  CREDIT_LI

In [8]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> pd.DataFrame:
    """
    Train several classification models on the training data and return a
    DataFrame summarising the best hyperparameters found for each model.

    Models trained:
        - Logistic Regression (baseline, default params)
        - Random Forest (RandomizedSearchCV)
        - Decision Tree (RandomizedSearchCV)
        - KNN (RandomizedSearchCV)
        - Gaussian Naive Bayes (GridSearchCV)

    Parameters
    ----------
    X_train : pd.DataFrame
        Training feature matrix.
    y_train : pd.Series
        Training target labels.

    Returns
    -------
    pd.DataFrame
        DataFrame with one row per model containing the model name and the
        best hyperparameters as a list in the 'best_params' column.
    """
    results = []

    # ---------------------------------------------------------------
    # 1. Logistic Regression (baseline - no tuning)
    # ---------------------------------------------------------------
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train, y_train)
    results.append({
        "model": "Logistic Regression",
        "best_params": list(log_reg.get_params().keys())
    })

    # ---------------------------------------------------------------
    # 2. Random Forest with RandomizedSearchCV
    # ---------------------------------------------------------------
    rf_param_dist = {
        "n_estimators": [50, 100, 200, 400],
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["auto", "sqrt"]
    }
    rf_cv = RandomizedSearchCV(
        RandomForestClassifier(random_state=42),
        param_distributions=rf_param_dist,
        n_iter=30,
        cv=3,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1,
    )
    rf_cv.fit(X_train, y_train)
    results.append({
        "model": "Random Forest",
        "best_params": list(rf_cv.best_params_.items())
    })

    # ---------------------------------------------------------------
    # 3. Decision Tree with RandomizedSearchCV
    # ---------------------------------------------------------------
    dt_param_dist = {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8],
        "max_features": ["auto", "sqrt", "log2"]
    }
    dt_cv = RandomizedSearchCV(
        DecisionTreeClassifier(random_state=42),
        param_distributions=dt_param_dist,
        n_iter=30,
        cv=3,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1,
    )
    dt_cv.fit(X_train, y_train)
    results.append({
        "model": "Decision Tree",
        "best_params": list(dt_cv.best_params_.items())
    })

    # ---------------------------------------------------------------
    # 4. KNN with RandomizedSearchCV
    # ---------------------------------------------------------------
    knn_param_dist = {
        "n_neighbors": list(range(1, 31)),
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan", "minkowski"],
        "p": [1, 2]
    }
    knn_cv = RandomizedSearchCV(
        KNeighborsClassifier(),
        param_distributions=knn_param_dist,
        n_iter=30,
        cv=3,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1,
    )
    knn_cv.fit(X_train, y_train)
    results.append({
        "model": "KNN",
        "best_params": list(knn_cv.best_params_.items())
    })

    # ---------------------------------------------------------------
    # 5. Gaussian Naive Bayes with GridSearchCV
    # ---------------------------------------------------------------
    nb_param_grid = {
        "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
    }
    nb_cv = GridSearchCV(
        GaussianNB(),
        param_grid=nb_param_grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
    )
    nb_cv.fit(X_train, y_train)
    results.append({
        "model": "Naive Bayes",
        "best_params": list(nb_cv.best_params_.items())
    })

    # ---------------------------------------------------------------
    # Assemble results into a DataFrame
    # ---------------------------------------------------------------
    return pd.DataFrame(results)

### **Model Training**

In [9]:
train_params_df = train_models(X_train, y_train)

2026/09/23 12:52:44 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '7dcbc752d8464a78b2afdbf159cf1d2f', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 12:52:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/23 12:52:49 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'a3ea421721a4423f8609ad44af0ff164', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 12:53:01 WARNING mlflow.sklearn: Saving scikit-learn models in the

In [10]:
print(train_params_df)

train_params_df.to_csv('train_params_df.csv', index=False)

                 model                                        best_params
0  Logistic Regression  [C, class_weight, dual, fit_intercept, interce...
1        Random Forest  [(n_estimators, 200), (min_samples_split, 5), ...
2        Decision Tree  [(min_samples_split, 5), (min_samples_leaf, 1)...
3                  KNN  [(weights, distance), (p, 1), (n_neighbors, 15...
4          Naive Bayes                           [(var_smoothing, 1e-09)]


#### **Logistic Regression**

In [11]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
log_reg_y_pred = logreg.predict(X_test)
print(classification_report(y_test, log_reg_y_pred))

2026/09/23 12:53:37 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '52f9527e35de489287b2871eacd26f83', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/09/23 12:53:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.77      0.87      0.82       345
           1       0.65      0.78      0.71       165
           2       0.82      0.80      0.81       123
           3       0.69      0.50      0.58       176
           4       0.65      0.86      0.74       131
           5       0.40      0.03      0.05        69

    accuracy                           0.72      1009
   macro avg       0.66      0.64      0.62      1009
weighted avg       0.70      0.72      0.70      1009



#### **KNN**

In [12]:
knn = KNeighborsClassifier(weights='distance', p=1, n_neighbors=15, metric='minkowski')
knn.fit(X_train, y_train)
knn_y_pred = knn.predict(X_test)
print(classification_report(y_test, knn_y_pred))

2026/09/23 12:53:43 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'ace9614519a24e0183ba0a86ee540d34', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/09/23 12:53:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.72      0.87      0.79       345
           1       0.62      0.58      0.60       165
           2       0.80      0.74      0.77       123
           3       0.62      0.43      0.51       176
           4       0.57      0.78      0.66       131
           5       0.71      0.25      0.37        69

    accuracy                           0.67      1009
   macro avg       0.67      0.61      0.61      1009
weighted avg       0.68      0.67      0.66      1009



#### **Naive Bayes**

In [13]:
nb = GaussianNB(var_smoothing=1e-09)
nb.fit(X_train, y_train)
nb_y_pred = nb.predict(X_test)
print(classification_report(y_test, nb_y_pred))

2026/09/23 12:53:48 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '7c0fbcd336c14e899b28f070437333aa', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 12:53:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.94      0.61      0.74       345
           1       0.52      0.85      0.65       165
           2       0.81      0.67      0.73       123
           3       0.69      0.62      0.65       176
           4       0.65      0.92      0.76       131
           5       0.90      0.91      0.91        69

    accuracy                           0.72      1009
   macro avg       0.75      0.76      0.74      1009
weighted avg       0.77      0.72      0.72      1009



#### **Decision Tree**

In [14]:
dec_tree = DecisionTreeClassifier(min_samples_split=5, min_samples_leaf=1, max_features='log2', 
                                  max_depth=None, criterion='gini', random_state=42)

dec_tree.fit(X_train, y_train)
dec_tree_y_pred = dec_tree.predict(X_test)
print(classification_report(y_test, dec_tree_y_pred))

2026/09/23 12:53:52 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2b657fa82a404819a8de3f266a1cd231', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 12:53:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.91      0.90      0.91       345
           1       0.89      0.85      0.87       165
           2       0.80      0.83      0.81       123
           3       0.81      0.82      0.81       176
           4       0.84      0.86      0.85       131
           5       0.78      0.81      0.79        69

    accuracy                           0.86      1009
   macro avg       0.84      0.85      0.84      1009
weighted avg       0.86      0.86      0.86      1009



#### **Random Forest**

In [15]:
rf = RandomForestClassifier(n_estimators=200, min_samples_split=5, min_samples_leaf=1,
                            max_features='sqrt', max_depth=None, random_state=42)

rf.fit(X_train, y_train)
rf_y_pred = rf.predict(X_test)
print(classification_report(y_test, rf_y_pred))

2026/09/23 12:53:57 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '30d8cde1a1934c65ac27b546da361afc', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 12:53:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.95      0.94      0.95       345
           1       0.94      0.91      0.93       165
           2       0.90      0.91      0.91       123
           3       0.90      0.91      0.91       176
           4       0.91      0.92      0.91       131
           5       0.92      0.97      0.94        69

    accuracy                           0.93      1009
   macro avg       0.92      0.93      0.92      1009
weighted avg       0.93      0.93      0.93      1009



### **Model Training (undersampling)**
---

In [20]:
from imblearn.under_sampling import RandomUnderSampler

In [23]:
undersampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = undersampler.fit_resample(X_train, y_train)

print(f"X_train shape: {X_resampled.shape}, y_train shape: {y_resampled.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

print(y_resampled.value_counts())

X_train shape: (1656, 17), y_train shape: (1656,)
X_test shape: (1009, 17), y_test shape: (1009,)
kmeans_labels
0    276
1    276
2    276
3    276
4    276
5    276
Name: count, dtype: int64


In [24]:
train_params_df = train_models(X_resampled, y_resampled)

print(train_params_df)

2026/09/23 13:26:33 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0ca4a6ff84bb436da4aaa1efb35127ff', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:26:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/23 13:26:38 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '70e207e2983140d1a2546162e89a4db8', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:26:48 WARNING mlflow.sklearn: Saving scikit-learn models in the

                 model                                        best_params
0  Logistic Regression  [C, class_weight, dual, fit_intercept, interce...
1        Random Forest  [(n_estimators, 400), (min_samples_split, 2), ...
2        Decision Tree  [(min_samples_split, 10), (min_samples_leaf, 8...
3                  KNN  [(weights, distance), (p, 2), (n_neighbors, 21...
4          Naive Bayes                           [(var_smoothing, 1e-09)]


In [25]:
train_params_df.to_csv('under_sampled_train_params_df.csv', index=False)

In [32]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_resampled, y_resampled)
log_reg_y_pred = logreg.predict(X_test)
print(classification_report(y_test, log_reg_y_pred))

2026/09/23 13:46:50 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6e722393f75d48a2bf4a95ee9ca18645', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:46:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.87      0.67      0.76       345
           1       0.57      0.78      0.66       165
           2       0.83      0.82      0.82       123
           3       0.59      0.58      0.59       176
           4       0.64      0.87      0.74       131
           5       0.30      0.19      0.23        69

    accuracy                           0.68      1009
   macro avg       0.63      0.65      0.63      1009
weighted avg       0.70      0.68      0.68      1009



In [33]:
knn = KNeighborsClassifier(weights='distance', p=2, n_neighbors=21, metric='manhattan')
knn.fit(X_resampled, y_resampled)
knn_y_pred = knn.predict(X_test)
print(classification_report(y_test, knn_y_pred))

2026/09/23 13:46:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '89c8db83ec774c3186cc129e450c5776', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:46:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.85      0.58      0.69       345
           1       0.50      0.71      0.59       165
           2       0.74      0.76      0.75       123
           3       0.62      0.38      0.47       176
           4       0.53      0.76      0.63       131
           5       0.31      0.52      0.39        69

    accuracy                           0.61      1009
   macro avg       0.59      0.62      0.59      1009
weighted avg       0.66      0.61      0.61      1009



In [34]:
nb = GaussianNB(var_smoothing=1e-09)
nb.fit(X_resampled, y_resampled)
nb_y_pred = nb.predict(X_test)
print(classification_report(y_test, nb_y_pred))

2026/09/23 13:47:00 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '72313993355545e18eae2a0c53817fac', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:47:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.96      0.59      0.73       345
           1       0.52      0.84      0.64       165
           2       0.77      0.68      0.72       123
           3       0.67      0.61      0.64       176
           4       0.63      0.92      0.75       131
           5       0.94      0.90      0.92        69

    accuracy                           0.71      1009
   macro avg       0.75      0.76      0.73      1009
weighted avg       0.77      0.71      0.72      1009



In [35]:
dec_tree = DecisionTreeClassifier(min_samples_split=10, min_samples_leaf=8, max_features='sqrt', 
                                  max_depth=20, criterion='entropy', random_state=42)

dec_tree.fit(X_resampled, y_resampled)
dec_tree_y_pred = dec_tree.predict(X_test)
print(classification_report(y_test, dec_tree_y_pred))

2026/09/23 13:47:04 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '7968d1649aca4d50af42a2ffe6b1dd73', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:47:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.95      0.82      0.88       345
           1       0.84      0.85      0.84       165
           2       0.73      0.80      0.76       123
           3       0.74      0.80      0.77       176
           4       0.81      0.86      0.84       131
           5       0.78      0.91      0.84        69

    accuracy                           0.83      1009
   macro avg       0.81      0.84      0.82      1009
weighted avg       0.84      0.83      0.83      1009



In [36]:
rf = RandomForestClassifier(n_estimators=400, min_samples_split=2, min_samples_leaf=2,
                            max_features='sqrt', max_depth=30, random_state=42)

rf.fit(X_resampled, y_resampled)
rf_y_pred = rf.predict(X_test)
print(classification_report(y_test, rf_y_pred))

2026/09/23 13:47:09 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '047cc834c84340f7a583bec5ea852daf', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/09/23 13:47:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.98      0.91      0.94       345
           1       0.91      0.92      0.91       165
           2       0.88      0.92      0.90       123
           3       0.88      0.90      0.89       176
           4       0.90      0.93      0.91       131
           5       0.89      0.99      0.94        69

    accuracy                           0.92      1009
   macro avg       0.91      0.93      0.92      1009
weighted avg       0.92      0.92      0.92      1009



Consistently across all models, Under sampling is WORSE for the minority classes (classes that are not class 0)

### **Model Training (Oversampling)**
---

In [37]:
from imblearn.over_sampling import RandomOverSampler
oversampler = RandomOverSampler(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = oversampler.fit_resample(X_train, y_train)

In [38]:
print(f"X_train shape: {X_resampled.shape}, y_train shape: {y_resampled.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

print(y_resampled.value_counts())

X_train shape: (8286, 17), y_train shape: (8286,)
X_test shape: (1009, 17), y_test shape: (1009,)
kmeans_labels
3    1381
4    1381
2    1381
0    1381
1    1381
5    1381
Name: count, dtype: int64


In [39]:
train_params_df = train_models(X_resampled, y_resampled)

print(train_params_df)

2026/09/23 13:56:11 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8eaf019fa1b746559f9f575f50a6fdb1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:56:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/23 13:56:18 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6ca7d78650ac4e66bd67927c67f5841b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:56:40 WARNING mlflow.sklearn: Saving scikit-learn models in the

                 model                                        best_params
0  Logistic Regression  [C, class_weight, dual, fit_intercept, interce...
1        Random Forest  [(n_estimators, 200), (min_samples_split, 5), ...
2        Decision Tree  [(min_samples_split, 5), (min_samples_leaf, 1)...
3                  KNN  [(weights, distance), (p, 2), (n_neighbors, 5)...
4          Naive Bayes                           [(var_smoothing, 1e-09)]


In [40]:
train_params_df.to_csv('oversampled_train_params_df.csv', index=False)

In [41]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_resampled, y_resampled)
log_reg_y_pred = logreg.predict(X_test)
print(classification_report(y_test, log_reg_y_pred))

2026/09/23 13:57:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '157df3399e99476b965a7f81b4b50408', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:57:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.84      0.70      0.76       345
           1       0.53      0.84      0.65       165
           2       0.77      0.83      0.80       123
           3       0.67      0.49      0.57       176
           4       0.69      0.87      0.77       131
           5       0.27      0.14      0.19        69

    accuracy                           0.69      1009
   macro avg       0.63      0.65      0.62      1009
weighted avg       0.69      0.69      0.68      1009



In [42]:
knn = KNeighborsClassifier(weights='distance', p=2, n_neighbors=5, metric='manhattan')
knn.fit(X_resampled, y_resampled)
knn_y_pred = knn.predict(X_test)
print(classification_report(y_test, knn_y_pred))

2026/09/23 13:58:35 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '30c2bba64a474a42bb94924d85dd9180', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:58:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.80      0.67      0.73       345
           1       0.53      0.61      0.56       165
           2       0.74      0.80      0.77       123
           3       0.53      0.45      0.49       176
           4       0.59      0.66      0.62       131
           5       0.31      0.46      0.37        69

    accuracy                           0.62      1009
   macro avg       0.58      0.61      0.59      1009
weighted avg       0.64      0.62      0.63      1009



In [43]:
nb = GaussianNB(var_smoothing=1e-09)
nb.fit(X_resampled, y_resampled)
nb_y_pred = nb.predict(X_test)
print(classification_report(y_test, nb_y_pred))

2026/09/23 13:58:51 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8868c0039b0b4c9a94fa5bd82b5831d6', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:58:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.94      0.59      0.72       345
           1       0.51      0.86      0.64       165
           2       0.81      0.68      0.74       123
           3       0.69      0.62      0.65       176
           4       0.66      0.92      0.76       131
           5       0.89      0.93      0.91        69

    accuracy                           0.72      1009
   macro avg       0.75      0.77      0.74      1009
weighted avg       0.77      0.72      0.72      1009



In [44]:
dec_tree = DecisionTreeClassifier(min_samples_split=5, min_samples_leaf=1, max_features='log2', 
                                  max_depth=None, criterion='gini', random_state=42)

dec_tree.fit(X_resampled, y_resampled)
dec_tree_y_pred = dec_tree.predict(X_test)
print(classification_report(y_test, dec_tree_y_pred))

2026/09/23 13:59:49 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8dbe9ed2d01b4ac99fc6e6a8f3600860', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 13:59:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.92      0.92      0.92       345
           1       0.85      0.85      0.85       165
           2       0.81      0.82      0.82       123
           3       0.81      0.79      0.80       176
           4       0.85      0.84      0.85       131
           5       0.86      0.93      0.90        69

    accuracy                           0.86      1009
   macro avg       0.85      0.86      0.85      1009
weighted avg       0.86      0.86      0.86      1009



In [45]:
rf = RandomForestClassifier(n_estimators=200, min_samples_split=5, min_samples_leaf=1,
                            max_features='sqrt', max_depth=None, random_state=42)

rf.fit(X_resampled, y_resampled)
rf_y_pred = rf.predict(X_test)
print(classification_report(y_test, rf_y_pred))

2026/09/23 14:00:28 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '40b52e17cf8149d1a4f15d3cb9310574', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/09/23 14:00:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


              precision    recall  f1-score   support

           0       0.97      0.94      0.95       345
           1       0.94      0.94      0.94       165
           2       0.91      0.94      0.93       123
           3       0.91      0.90      0.91       176
           4       0.92      0.93      0.92       131
           5       0.89      0.99      0.94        69

    accuracy                           0.93      1009
   macro avg       0.92      0.94      0.93      1009
weighted avg       0.94      0.93      0.93      1009



Best metrics:
Oversampled Random Forest 

---
### **5. Performance Metrics**


---
### **6. Conclusion**
